# D03-信令数据洞察

In [ ]:
%reload_ext autoreload
%autoreload 2 
# 上面这俩用于自动重载

import sqlite3 # 先导入内置的模块

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from signal_insight import config, quality, viz
from signal_insight.viz import (MAIN,MAIN_LIGHT,MAIN_DARK,MAIN_MID,GRAY,
                    GRAY_DARK,RED,AMBER,INK,MUTED,FAINT) # 配色常量

viz.apply_theme(theme="default")

con = sqlite3.connect(config.DB_PATH) # 连接数据库
DAY = config.ANALYSIS_DAY
NB = "D03_信令数据洞察"
SOURCE = f"{config.SOURCE_NOTE}:脚本:{NB} 制图:{config.AUTHOR}"

In [ ]:
raw = pd.read_sql("SELECT * FROM ods_signal", con)

raw['ts'] = pd.to_datetime(raw['timestamp'], unit="ms") + \
    pd.Timedelta(hours = config.UTC_OFFSET_HOURS)
raw.shape

## 完整性分析：那些字段有缺失
需求：每个字段缺多少？缺的字段要不要删除，缺值的行要不要删

In [ ]:
missing = pd.DataFrame({
    "缺失数":
    raw.drop(columns='ts').isna().sum() + raw.drop(columns="ts").apply(
        lambda x: x.astype('string').str.strip().eq("").sum() if
        (pd.api.types.is_string_dtype(x) or x.dtype == object) else 0)
}).rename_axis('字段').reset_index()
missing['缺失率'] = round(missing['缺失数'] / len(raw),7)
missing = missing.sort_values(by='缺失率').reset_index(drop=True)
missing.to_sql("ads_missing_by_field", con, if_exists="replace", index=False)
missing

In [ ]:
m = pd.read_sql("SELECT * FROM ads_missing_by_field", con)
full = m[m['缺失率'] == 1]  # 正列为空的字段
worst_other = m.loc[m['缺失率'] < 1, '缺失数'].max()

fig, ax = plt.subplots(figsize=(10, 5.6), dpi=100)
colors = [RED if r == 1 else GRAY for r in m['缺失率']]
bars = ax.barh(m['字段'], m['缺失率'], color=colors, height=.62)
labels = [
    '100%(整列为空)' if r == 1 else (f"{n}行" if n > 0 else "0")
    for r, n in zip(m['缺失率'], m['缺失数'])
]

ax.bar_label(bars, labels = labels, padding = 4, color = INK)
ax.set_xlim(0, 1.25)
ax.grid()
title = f"{', '.join(sorted(full['字段']))} 整列为空，其余字段缺失最多为{worst_other}"
viz.finish(fig, title, 
    f"各字段缺失率：空值和空白字符串都算缺失，口径：ods_signal全部{len(raw):,}行",
    SOURCE,"D3_01_字段缺失率.png"
    )

- 完整性检查我们发现ods_single的tmp0, tmp1两个字段整列为空，也就是有532,828行。全部缺失，那对于这种直接删除即可，关键字段imsi有15行缺失，lac_id有14行有4行缺失。cell_id有5行缺失，合计占比不到0.01%，所以可以删除对应的记录。那这个可以看图第20第301检查像我们的DQ_02以及DQ_03，其余看不见，不是没画，而是比例太小了啊

In [ ]:
quality.dq_record(con, "DQ_02","完整性",
    "整列为空的字段","ods_signal.tmp0 / tmp1", "2个字段 100%为空",'不通过',
    evidence="tmp0 全部为空值，tmp1全部为空格", decision="删除这两个字段"
    )

quality.dq_record(con, "DQ_03","完整性",
    "关键字段缺失","ods_signal.imsi / lac_id / cell_id",
    f"{raw[['imsi', 'lac_id','cell_id']].isna().any(axis=1).sum()}行","通过",
    evidence="imsi缺15行，lac_id缺4, cell_id缺5，占比不到0.01%",
    decision="删除这些行"
    )

## 规范性：IMSI，手机号格式对不对

需求：IMSI应该是18位纯数字，手机号脱敏后应该是14位，检查下有没有不符合的

In [ ]:
ids = raw['imsi'].dropna()
bad_imsi = raw[raw['imsi'].notna()
        & ~raw['imsi'].str.fullmatch(r"\d{18}", na=False)]
print(f"IMSI非空 {len(ids):,}行，其中长度为18的 {ids.str.len().eq(18).sum():,}行")
print(f"不是 18位纯数字的：{len(bad_imsi)}行")
bad_imsi[['imsi','phone','lac_id','cell_id','nid']]

In [ ]:
phone_len = raw['phone'].str.len().value_counts()
odd_phone = raw.loc[raw['phone'].str.len() != 14,'phone'].value_counts()
print(f"手机号长度分布： {phone_len.to_dict()}")
print(f'异常手机号：{odd_phone.to_dict()}，\
涉及 {raw.loc[raw["phone"] == "6666",'imsi'].nunique()}个 IMSI')

In [ ]:
quality.dq_record(con,
                  "DQ-04",
                  "规范性",
                  "IMSI 格式",
                  "ods_signal.imsi",
                  f"{len(bad_imsi)} 行含 * ^ # 等掩码字符",
                  "警告",
                  evidence="长度都是 18，但不是纯数字，疑似脱敏时误处理",
                  decision="删除这些行，并反馈数据提供方")
quality.dq_record(con,
                  "DQ-05",
                  "规范性",
                  "手机号格式",
                  "ods_signal.phone",
                  "36 行为占位值 6666（14 个 IMSI）",
                  "警告",
                  evidence="其余手机号均为 14 位",
                  decision="手机号不作为主键使用；分析中不需要手机号，明细表不保留")

## 唯一性：有没有完全重复的记录

需要：所有字段一模一样的记录有多少

In [ ]:
raw.columns

In [ ]:
orig_cols = ['timestamp', 'imsi', 'lac_id', 'cell_id',
            'phone', 'timestamp1','nid', 'npid']

# 这个是根据业务作出决定，因为关键字段缺失了就没法定位，所以整行丢掉
step = raw.dropna(subset=['imsi','lac_id','cell_id'])
# 格式不对，比如IMSI有问题，就定位不到人了
step = step[step['imsi'].str.fullmatch(r"\d{18}")]
# 每组重复只保留第一条
dup_mask = step.duplicated(subset=orig_cols, keep="first")
print(f'完全重复：{dup_mask.sum():,}行,占比{dup_mask.mean():.2%}')

step[step.duplicated(subset=orig_cols,keep=False)].sort_values(orig_cols).head(5)

In [ ]:
quality.dq_record(con,
                  "DQ-06",
                  "唯一性",
                  "完全重复记录",
                  "ods_signal（全部原始字段）",
                  f"{dup_mask.sum():,} 行（{dup_mask.mean():.2%}）",
                  "警告",
                  evidence="所有字段完全相同，疑似采集或入库时重复写入",
                  decision="每组只保留一条")

## 一致性：两个时间戳差多少

两个都是时间戳，他们差多少，分析时用那个？常见的情形一个是开始时间，一个是写入时间，不过是猜测具体结论需要测量后再决定

In [ ]:
# 计算一下两者都时间差，单位是毫秒
lag = raw['timestamp'] - raw['timestamp1']
print(lag.describe(percentiles=[0.5, 0.9, 0.99]).round(0))
print(f"timestamp 早于 timestamp1 的：{(lag < 0).sum()}行")

In [ ]:
quality.dq_record(
    con,
    "DQ-07",
    "一致性",
    "两个时间戳的差",
    "ods_signal.timestamp / timestamp1",
    f"中位 {lag.median():.0f} 毫秒，99% 在 {lag.quantile(0.99) / 1000:.1f} 秒内",
    "通过",
    evidence=f"只有 {(lag < 0).sum()} 行为负数",
    decision="统一用 timestamp 作为事件时间，差异对分钟级分析没有影响")

## 主键验证:IMSI前15位能不能代表一个人

之前我们认为：IMSI是18位，前15位是用户，后3位是会话实例号，于是我们按前15位统计用户，这种说法对吗？

不要相信来路不明的说法，得用数据验证他，思路：如果前15个人真是同一个人，那么同一个前15为虾米的应该只有一个手机号

**主键错了，后续所有按人统计的结果就都是错的，所以主键验证永远排在分析之前**

In [ ]:
valid = raw.dropna(subset=['imsi'])
valid = valid[valid['imsi'].str.fullmatch(r"\d{18}")
              & (valid['phone'] != "6666")]
# 只看那个imsi 配过那个手机号，这组对应关系，同一对出现多少次不重要，所以要先去重
pairs = valid[['imsi','phone']].drop_duplicates()
# 裁剪出前15位，作为另外一种认证方式
pairs['imsi15']  = pairs['imsi'].str[:15]

per18 = pairs.groupby("imsi")['phone'].nunique()
per15 = pairs.groupby('imsi15')['phone'].nunique()

key_check = pd.DataFrame({
    "分组方式": ['按前15位',"按完整18位"],
    "ID个数": [per15.shape[0], per18.shape[0]],
    "对应多个手机号的ID": [(per15 > 1).sum(), (per18 > 1).sum()]
})
key_check['占比'] = (key_check['对应多个手机号的ID'] / key_check['ID个数']).round(3)
key_check.to_sql("ads_key_check", con, if_exists="replace",index=False)

In [ ]:
multi = per15[per15 > 1].index
affected = pairs.loc[pairs['imsi15'].isin(multi),'imsi'].nunique()
suffuix_kinds = pairs['imsi'].str[-3:].nunique()
print(f"对应多个手机号的前15位分组："
    f"{len(multi)}组，里面装着{affected:,}个不同的18位IMSI,"
    f"占全部用户的{affected / len(per18):.1%}")
print(f"后三位一共有{suffuix_kinds}种取值")

In [ ]:
k = pd.read_sql("SELECT * FROM ads_key_check", con)
rows = [(f"按前 15 位分组\n（{k.loc[0, 'ID个数']:,} 组）", 
         k.loc[0, "对应多个手机号的ID"],
         k.loc[0, "ID个数"], "组对应多个手机号"),
        (f"被牵连的用户\n（{len(per18):,} 个 18 位 IMSI）", affected, len(per18),
         "个用户会被错误合并"),
        (f"按完整 18 位\n（{k.loc[1, 'ID个数']:,} 个）", k.loc[1, "对应多个手机号的ID"],
         k.loc[1, "ID个数"], "个对应多个手机号")]
rows

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

FIG_NAME = "D03_02_IMSI截断冲突.png"  # 按当天的编号改
SOURCE = f"数据来源:沈阳移动信令样本(脱敏,2018年国庆)；制图: {config.AUTHOR}"

# ---- 数据：np.int64 转成 int，标签并成一行（标签改放在条上方）----
data = [(lab.replace("\n", ""), int(bad), int(tot), suf)
        for lab, bad, tot, suf in rows]
(_, grp_bad, grp_tot, _), (_, usr_bad, usr_tot, _), (_, full_bad, full_tot,
    _) = data


def pct(n, d):
    """第 6 节：一般取整；很小的值多留位数，不写 0.0%"""
    p = n / d * 100
    if p == 0:
        return "0%"
    return f"{p:.2g}%" if p < 1 else f"{p:.0f}%"


# ---- 标题、副标题：数字全部由代码算 ----
tail = "完整 18 位零冲突" if full_bad == 0 else f"完整 18 位仍有 {full_bad:,} 组冲突"
title = f"只取前 15 位会误合并 {usr_bad:,} 个用户，{tail}"
subtitle = (f"{full_tot:,} 个 18 位 IMSI 分别按前 15 位、完整 18 位分组；\n"
            f"一组对应 ≥2 个手机号即为冲突，冲突组里的每个 IMSI 都计为被牵连用户")

# ---- 图形：每行一根 100% 条，红色 = 出问题的部分，灰色 = 一对一的部分 ----
fig, ax = plt.subplots(figsize=(10, 5.6), dpi=100)
ys, H = [0, 1, 2], 0.32  # 第 3 行（解决办法）和上两行拉开

for y, (label, bad, tot, suffix) in zip(ys, data):
    ok = bad == 0
    ax.barh(y, 1, height=H, color=GRAY, linewidth=0)
    if bad:
        share = bad / tot
        ax.barh(y, share, height=H, color=RED, linewidth=0)
        ax.text(share - 0.012,
                y,
                pct(bad, tot),
                ha="right",
                va="center",
                color="white",
                fontsize=10,
                fontweight="bold")
    top = y - H / 2 - 0.07  # 条的上沿（y 轴倒置）
    ax.text(0,
            top,
            label,
            ha="left",
            va="bottom",
            fontsize=10.5,
            color=MAIN if ok else INK,
            fontweight="bold" if ok else "normal")
    ax.text(1,
            top,
            f"{bad:,} {suffix}",
            ha="right",
            va="bottom",
            fontsize=10.5,
            color=MAIN if ok else RED,
            fontweight="bold")

ax.axhline((ys[1] + ys[2]) / 2 - 0.12, color=GRAY, lw=0.8, ls=(0, (3, 3)))
ax.set_xlim(0, 1)
ax.set_ylim(ys[-1] + H / 2 + 0.05, ys[0] - H / 2 - 0.45)  # 倒置：第 1 行在最上
ax.axis("off")
# ---- 外框：强调色短条 + 标题 + 副标题 + 来源行（D02 起可换成 viz.finish）----
viz.finish(fig, title, subtitle, SOURCE, FIG_NAME)
plt.show()

弱按照IMSI的前15位识别用户，4667个分组中有729组(15.6%) 对应多个手机号，牵连1707 名用户 (30.2%)会被错误合并，完整的18位IMSI与手机号一一对应。因此用户主键采用完整的18位IMSI(图D3_02; DQ-08, '不通过')

In [ ]:
quality.dq_record(
    con,
    "DQ-08",
    "唯一性",
    "用户主键：IMSI 前 15 位能否代表一个人",
    "ods_signal.imsi",
    f"前 15 位有 {len(multi)} 组对应多个手机号（{len(multi) / len(per15):.1%}）",
    "不通过",
    evidence=
    f"这些分组牵连 {affected:,} 个 18 位 IMSI（{affected / len(per18):.1%}）；18 位 IMSI 与手机号一一对应",
    decision="用户主键改用完整 18 位 IMSI")

## 执行清洗：数据写回dwd_signal
需求：把前面的决定按照固定的顺序执行一遍，记下每一步剔除了多少行，最后只保留10月3日的数据，写到明细表里面dwd_signal, 注意明细表只保留分析要用的字段：imsi, ts, dt， lac_id, cell_id, lac_cell,nid，切记手机号这类用不到的敏感字段不进入明细表，**用不到的个人信息不保留，这是数据合规的基本要求**

- 务必逐步记录
    - 数据质量报告里的清洗说明就是这张表，不是事后回忆
    - 哪一步剔的异常多，通过可视化可以一眼看出来
    - 别人拿到同一份数据按照相同的处理机制能跑出出一模一样的数据，这就是可复现

In [ ]:
orig_cols = ['timestamp', 'imsi', 'lac_id', 'cell_id',
            'phone', 'timestamp1','nid', 'npid']
steps = []
df = raw.drop(columns=['tmp0','tmp1'])
n0 = len(df)

def log_step(name, before, after):
    steps.append({"步骤": name, "剔除行数":before - after,"剩余行数":after})
    
n = len(df); df = df.dropna(subset=["imsi", "lac_id", "cell_id"])
log_step("缺 imsi / lac / cell", n, len(df))

n = len(df); df = df[df['imsi'].str.fullmatch(r"\d{18}")]
log_step('IMSI 格式异常', n, len(df))

n = len(df); df = df.drop_duplicates(subset=orig_cols)
log_step("完全重复", n, len(df))

n = len(df); df = df[df['ts'].dt.strftime("%Y-%m-%d") == DAY]
log_step("不是10月3日", n, len(df))

steps = pd.DataFrame(steps)
steps['剔除占比'] = steps['剔除行数'] / n0
steps.to_sql("ads_clean_steps", con, if_exists="replace", index=False)
steps

::: IMPORTANT DWD明细层
DWD明细表就是data warehouse detail，它是对ODS表做清洗和规范化之后的明细数据。
:::

In [ ]:
dwd = pd.DataFrame({
    "imsi": df['imsi'],
    "ts": df['ts'],
    "dt": DAY,
    "lac_id": df['lac_id'].astype("int64"),
    "cell_id":df['cell_id'].astype("int64"),
    "nid": df['nid']
})
dwd["lac_cell"] = dwd['lac_id'].astype(str) + "-" + dwd['cell_id'].astype(str)

dwd = dwd.sort_values(['imsi','ts']).reset_index(drop=True)

dwd['ts'] = dwd['ts'].dt.strftime("%Y-%m-%d %H:%M:%S.f%").str[:-3]

dwd.to_sql("dwd_signal", con, if_exists='replace', index=False)

In [ ]:
n_db = pd.read_sql("SELECT COUNT(*) AS n FROM dwd_signal", con)['n'][0]
dwd.head()

In [ ]:
steps = pd.read_sql("SELECT * FROM ads_clean_steps", con)

In [ ]:
import math
from matplotlib.patches import ConnectionPatch

SOURCE = (f"数据来源：沈阳移动信令样本（脱敏，2018 年国庆）；"
          f"制图：{config.AUTHOR}")
FIG_NAME = "D03_04_清洗剔除明细.png"

# ---------- 1. 数字全部由 steps 计算，先核对口径 ----------
names = steps["步骤"].tolist()
removed_n = steps["剔除行数"].astype(int).tolist()
remain_n = steps["剩余行数"].astype(int).tolist()
raw = remain_n[0] + removed_n[0]                 # 原始行数
kept = remain_n[-1]                              # 清洗后保留
removed = raw - kept                             # 剔除合计
assert removed == sum(removed_n), "剔除行数之和 ≠ 原始 − 保留，steps 表口径不一致"
assert all(remain_n[i - 1] - removed_n[i] == remain_n[i]
           for i in range(1, len(names))), "剩余行数没有逐步递减，检查步骤顺序"

def fmt_pct(p, nd=None):
    """规范第 6 节：≥10% 取整；1–10% 一位小数；<1% 保留两位有效数字（不写 0.0%）。"""
    v = p * 100
    if v == 0:
        return "0%"
    if nd is None:
        nd = 0 if v >= 10 else 1 if v >= 1 else -math.floor(math.log10(v)) + 1
    return f"{v:.{nd}f}%"

NUM = "①②③④⑤⑥⑦⑧⑨"
LABEL = {"不是10月3日": "不是 10 月 3 日"}                # 图上的步骤名，其余沿用表里原名
PHRASE = {"缺 imsi / lac / cell": "缺关键字段的数据",      # 标题里的说法
          "IMSI 格式异常": "IMSI 格式异常的数据",
          "完全重复": "完全重复的数据",
          "不是10月3日": "非 10 月 3 日的数据"}

# 按“占全部剔除的份额”分三档：最多的一步 / ≥ 5% / 不到 1% 算可忽略
share_i = [n / removed for n in removed_n]
k = max(range(len(names)), key=lambda i: removed_n[i])
MID_MIN, NEG_MAX = 0.05, 0.01
tier = ["hi" if i == k else "mid" if s >= MID_MIN else "neg" if s < NEG_MAX else "low"
        for i, s in enumerate(share_i)]
color = [{"hi": MAIN, "mid": MAIN_MID}.get(t, GRAY) for t in tier]
cum = [sum(removed_n[:i]) for i in range(len(names) + 1)]  # 瀑布每一级的起点

# ---------- 2. 画布：左总量柱，右放大瀑布（只有这两个绘图区，不要再建别的 ax）----------
fig = plt.figure(figsize=(10, 5.6), dpi=100)                # 16:9，汇报 PPT 用
gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.55, 3.8])
ax_l, ax_r = fig.add_subplot(gs[0]), fig.add_subplot(gs[2])
TOP, BOT = 1.20, 0.12   # 两边纵轴都按“总量 × 比例”设上下限：柱顶和合计线同高；
                        # 0 线下方留位置写步骤名（写在绘图区内，不受 finish 边距影响）
HEAD = TOP * 0.92       # 两个小标题的高度
W_L, W_R = 0.72, 0.58

# --- 2a 左：原始总量一根柱，保留段浅主色，剔除段按步骤顺序叠在顶上（颜色与右图一致）---
ax_l.bar(0, kept, width=W_L, color=MAIN_LIGHT, lw=0)
for i, n in enumerate(removed_n):
    ax_l.bar(0, n, bottom=kept + cum[i], width=W_L, color=color[i], lw=0)
ax_l.set_xlim(-0.5, 0.5)
ax_l.set_ylim(-raw * BOT, raw * TOP)
ax_l.text(0, kept / 2, f"保留\n{kept:,} 行\n{fmt_pct(kept / raw, 1)}",
          ha="center", va="center", fontsize=11, color=INK, linespacing=1.6)
ax_l.text(-W_L / 2, raw * HEAD, f"原始 {raw:,} 行",
          ha="left", va="top", fontsize=11, fontweight="bold", color=INK)

# --- 2b 右：剔除的行按执行顺序累加（瀑布），纵轴从 0 开始 ---
x = list(range(len(names)))
ax_r.bar(x, removed_n, bottom=cum[:-1], width=W_R, color=color, lw=0)
for i in range(len(names) - 1):                             # 台阶之间的细连线
    ax_r.plot([i + W_R / 2, i + 1 - W_R / 2], [cum[i + 1]] * 2, color=GRAY, lw=0.8)
ax_r.axhline(removed, color=GRAY, lw=0.8, ls=(0, (3, 3)), zorder=0)   # 合计线
ax_r.set_xlim(-0.6, len(names) - 0.4)
ax_r.set_ylim(-removed * BOT, removed * TOP)
ax_r.text(len(names) - 0.4, removed * 1.02,
          f"合计剔除 {removed:,} 行（{fmt_pct(removed / raw, 1)}）",
          ha="right", va="bottom", fontsize=10.5, fontweight="bold", color=INK)
ax_r.text(-0.6, removed * HEAD, "其中剔除的部分，按执行顺序放大",
          ha="left", va="top", fontsize=11, fontweight="bold", color=INK)

# --- 2c 标注：剔除多的写全（行数、占原始、占剔除），可忽略的只写行数并合并说明 ---
for i, n in enumerate(removed_n):
    pct, sh = fmt_pct(n / raw), fmt_pct(share_i[i])
    if tier[i] == "hi":                                     # 最多的一步：写在柱子左侧，加粗
        ax_r.text(i - W_R / 2 - 0.06, cum[i] + n * 0.6,
                  f"{n:,} 行（{pct}）\n占全部剔除的 {sh}",
                  ha="right", va="center", fontsize=11, fontweight="bold",
                  color=MAIN, linespacing=1.5)
    elif tier[i] == "mid":                                  # 较多的：柱顶，主色
        ax_r.annotate(f"{n:,} 行（{pct}）\n占全部剔除的 {sh}", (i, cum[i + 1]),
                      xytext=(0, 5), textcoords="offset points", ha="center",
                      va="bottom", fontsize=10.5, color=MAIN, linespacing=1.4)
    else:                                                   # 很少的：只写行数，灰字
        ax_r.annotate(f"{n:,} 行", (i, cum[i + 1]), xytext=(0, 5),
                      textcoords="offset points", ha="center", va="bottom",
                      fontsize=10, color=MUTED)

neg = [i for i, t in enumerate(tier) if t == "neg"]
if len(neg) >= 2 and neg == list(range(neg[0], neg[-1] + 1)):   # 相邻的可忽略步骤：括号合并说明
    yb = removed * 0.13
    xa, xb = neg[0] - W_R / 2, neg[-1] + W_R / 2
    ax_r.plot([xa, xa, xb, xb], [yb - removed * 0.025, yb, yb, yb - removed * 0.025],
              color=GRAY, lw=0.8)
    n_neg = sum(removed_n[i] for i in neg)
    ax_r.text((xa + xb) / 2, yb + removed * 0.02,
              f"{len(neg)} 步合计 {n_neg:,} 行，占全部剔除的 {fmt_pct(n_neg / removed)}，可以忽略",
              ha="center", va="bottom", fontsize=10, color=MUTED)

for i, s in enumerate(names):                               # 步骤名写在 0 线下方
    ax_r.text(i, -removed * 0.04, f"{NUM[i] if i < len(NUM) else i + 1} {LABEL.get(s, s)}",
              ha="center", va="top", fontsize=10.5,
              color=MAIN if tier[i] in ("hi", "mid") else MUTED,
              fontweight="bold" if tier[i] == "hi" else "normal")

# --- 2d 样式：不要坐标轴、刻度和网格（数字直接标在图上），只留 0 基线 ---
for a in (ax_l, ax_r):
    a.set_axis_off()
ax_l.plot([-W_L / 2 - 0.08, W_L / 2], [0, 0], color=GRAY, lw=0.8)   # 左边基线只比柱子宽一点
ax_r.axhline(0, color=GRAY, lw=0.8)

# --- 2e 放大线：左柱的剔除段（kept→raw）对应右图的 0→合计 ---
for ya, yb in ((raw, removed), (kept, 0)):
    fig.add_artist(ConnectionPatch(
        xyA=(W_L / 2, ya), coordsA=ax_l.transData,
        xyB=(-0.6, yb), coordsB=ax_r.transData,
        color=GRAY, lw=0.8, zorder=5))

# ---------- 3. 标题、副标题、来源：交给 viz.finish ----------
title = (f"清洗剔除 {fmt_pct(removed / raw, 1)} 的记录，"
         f"其中 {fmt_pct(share_i[k])} 是{PHRASE.get(names[k], '“' + names[k] + '”')}")
subtitle = (f"原始 {raw:,} 行按 ①→{NUM[len(names) - 1]} 依次清洗，每行只计入第一个命中的步骤\n"
            "右图是左柱顶部剔除段的放大，两边刻度不同；括号里的百分比以原始行数为分母")

viz.finish(fig, title, subtitle, SOURCE, FIG_NAME)
plt.show()

## 更新数据字典与发现清单
| 字段 | 含义 | 可信度 | 依据 |
|---|---|---|---|
| timestamp | 事件时间，UTC 毫秒；+8 小时为北京时间 | 已验证 | 小时分布符合作息（D1） |
| imsi | 用户标识，完整 18 位才是一个人 | 已验证 | 与手机号一一对应；前 15 位不成立（2.5） |
| lac_id / cell_id | 位置区 / 小区编号 | 推测 | D3 与基站表关联验证 |
| phone | 脱敏手机号 | 已验证 | 14 位；6666 为占位值；明细表不保留 |
| timestamp1 | 第二个时间戳，与 timestamp 中位相差 23 毫秒 | 已验证 | 不使用 |
| nid | 事件类型编码 | 推测（字典不可信） | 见 2.8 选讲 |

---

| 编号 | 发现 | 证据 | 影响 / 决定 |
|---|---|---|---|
| F5 | tmp0、tmp1 整列为空，其余字段最多缺 15 行 | 图 D2_01 | 删两列；缺关键字段的 24 行删除 |
| F6 | 8 条 IMSI 含 * ^ # 等掩码字符 | bad_imsi | 删除，反馈数据提供方 |
| F7 | 完全重复 2,568 行（0.48%） | 2.3 | 每组保留一条 |
| F8 | 按前 15 位分组，15.6% 的“用户”对应多个手机号，牵连 30.2% 的用户 | 图 D2_02 | 主键改用 18 位 IMSI |
| F9 | 清洗共剔除 3.2%，其中 2.7 个百分点是非 10 月 3 日的记录 | 图 D2_03 | 明细表 515,931 行、用户 5,644 个 |

## nid字段是什么样子
可以尝试用数据反推nid, 思路如果某个nid代表位置更新类事件，那么它出现是，本人的位置区LAC 应该比 平常更容易和上一条不一样，把各个nid的伴随位置区变化的比例排出来看一下就知道了

In [ ]:
x = dwd[['imsi','ts','lac_id','nid']].copy()
prev_lac = x.groupby("imsi")['lac_id'].shift()

x["LAC变化"] = prev_lac.notna() & (x['lac_id'] != prev_lac)
x = x[prev_lac.notna()]
by_nid = x.groupby('nid').agg(
    记录数= ("LAC变化", "size"),
    LAC变化占比=("LAC变化", "mean")
)

top12 = by_nid.nlargest(12, '记录数').sort_values("LAC变化占比")
overall = x['LAC变化'].mean()
top12.to_sql("ads_nid_lac_change", con, if_exists='replace')

In [ ]:
import matplotlib.pyplot as plt
from signal_insight import viz

HIGH = 0.18  # “明显伴随位置区变化”的阈值
best = top12['LAC变化占比'].idxmax()
d = top12.sort_values("LAC变化占比")  # 从下往上递增，最高的在顶部
r = d["LAC变化占比"]
colors = [viz.MAIN if v >= HIGH else viz.MAIN_MID if v > overall else viz.GRAY for v in r]

fig, ax = plt.subplots(figsize=(10, 5.6),dpi=100)
y = range(len(d))
ax.barh(y, r, color=colors, height=0.78)
ax.set_yticks(list(y), [f"nid {n}" for n in d.index], fontsize=10.5)
ax.set_xlim(0, .6)  # 条形从 0 开始
ax.set_xticklabels([])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.tick_params(axis="x", labelsize=10, colors=viz.MUTED)
ax.tick_params(axis="y", length=0, colors=viz.INK)
for s in ("top", "right", "left"):
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color(viz.GRAY)

# 条尾直接标注：占比 + 记录数（替代图例）
for i, (v, n) in enumerate(zip(r, d["记录数"])):
    ax.text(v + 0.01, i, f"{v:.0%}", va="center", fontsize=10, color=viz.INK,
            fontweight="bold" if v >= HIGH else "normal",
            bbox=dict(fc="white", ec="none", pad=1))
    ax.text(v + 0.045, i, f"{n:,} 条", va="center", fontsize=9, color=viz.MUTED,
            bbox=dict(fc="white", ec="none", pad=1))

# 整体基准线
ax.axvline(overall, color=viz.MUTED, ls="--", lw=1, zorder=0)
ax.text(overall, len(d) - 0.35, f"全部记录 {overall:.0%}", ha="center", va="bottom",
        fontsize=9.5, color=viz.MUTED)

hi = d[r >= HIGH].sort_values("LAC变化占比", ascending=False)
title = (f"nid {'、'.join(map(str, hi.index))} 伴随位置区变化 "
      f"{hi['LAC变化占比'].min():.0%}–{hi['LAC变化占比'].max():.0%}，整体仅 {overall:.0%}")
subtitle = (f"记录数前 12 的 nid（共 {d['记录数'].sum():,} 条）；按用户、时间排序，比较每条记录的 LAC 与本人上一条是否不同。\n"
       f"深蓝 ≥ {HIGH:.0%}，像位置更新类事件；灰色低于整体，不随位置区变化。只说明相关，两条记录间可能隔很久")
source= f"数据来源：沈阳移动信令样本（脱敏，2018 年国庆）；制表：{config.AUTHOR}"


viz.finish(fig, title, subtitle, SOURCE, "D05_03_nid位置区变化.png")
plt.show()

对 nid 字段的行为验证表明，nid=5 事件有 51% 伴随位置区变化，远高于全部记录的 18%，行为最接近“位置更新”；而字典将其标为会话类事件，标为“位置更新完成”的 nid=2 仅为 12%。字典与数据行为不一致，nid 含义标为“未知”，分析中不使用（图 D2_04；DQ-09）。

In [ ]:
mapping = pd.read_sql("SELECT * FROM ods_nid_mapping",con)
mapping.head()

::: CAUTION 警告
来路不明的数据字典（包括网上搜的、大模型“推测”的）不能直接用。能用数据检验的先检验，检验不了的标“未知”，并向数据提供方要正式的接口文档。 所以不要无脑使用大模型.
:::

In [ ]:
quality.dq_record(
    con,
    "DQ-09",
    "准确性",
    "nid 字典与数据行为是否一致",
    "ods_nid_mapping",
    f"nid={best} 有 {top12.loc[best, 'LAC变化占比']:.0%} 伴随位置区变化，字典却标为会话类事件",
    "不通过",
    evidence=
    f"全部记录平均 {overall:.0%}；字典标为“位置更新”的 nid=2 只有 {by_nid.loc[2, 'LAC变化占比']:.0%}",
    decision="字典不采用；nid 含义标为未知，向数据提供方索取接口文档")

## 两个时间戳的差绘制成分布

In [ ]:
# ============================================================
# lag 分布：累计分布（ECDF，对数横轴）+ 分段计数
# 前提：notebook 里已有 lag、viz、config 和配色常量
#       MAIN / MAIN_LIGHT / MAIN_MID / GRAY / INK / MUTED / FAINT
# 口径：差值小于 0 的记录不绘制、不参与统计，行数在副标题说明
from matplotlib.ticker import FixedLocator, FixedFormatter, PercentFormatter

# ---- 使用前确认的三处 ----------------------------------------
LAG_TO_SEC = 1                     # lag 原始单位换算成秒；若是毫秒改成 0.001
LAG_DEF = "两个时间戳之差"          # 改成实际口径，例如“入库时间 − 采集时间”
FIG_NAME = "D03_05_lag分布.png"     # 进报告时按规范第 8 节改成 Q0x_lag分布.png
SOURCE = (f"数据来源：沈阳移动信令样本（脱敏，2018 年国庆）；"
          f"制图：{config.AUTHOR}")   # 有版本号的话补上：；配置 {config.VERSION}


# ---- 格式化（规范第 6 节）-------------------------------------
def fmt_dur(sec):
    """秒 → 人读的时长；超过 2 小时改用小时。"""
    a, sign = abs(sec), ("−" if sec < 0 else "")
    if a < 60:
        txt = f"{a:.0f} 秒"
    elif a < 7200:
        m = a / 60
        txt = f"{m:.1f} 分钟" if m < 10 else f"{m:.0f} 分钟"
    else:
        h = a / 3600
        txt = f"{h:.1f} 小时" if h < 100 else f"{h:,.0f} 小时"
    return sign + txt


def fmt_pct(p):
    """≥10% 取整，1–10% 一位小数，<1% 保留两位有效数字（不会出现 0.0%）。"""
    v = p * 100
    if v == 0:
        return "0%"
    if v >= 10:
        return f"{v:.0f}%"
    if v >= 1:
        return f"{v:.1f}%"
    d = -int(np.floor(np.log10(v))) + 1
    return f"{v:.{d}f}%"


# ---- 统计量：全部由数据计算 -----------------------------------
s_all = pd.to_numeric(lag, errors="coerce").dropna() * LAG_TO_SEC
neg = int((s_all < 0).sum())                 # 差值小于 0：不绘制，只在副标题说明
s = s_all[s_all >= 0]
n = len(s)
assert n + neg == len(s_all)
p50, p90, p99 = s.quantile([0.50, 0.90, 0.99])
mean = s.mean()
mean_rank = (s <= mean).mean()              # 均值落在第几百分位
z0 = (s == 0).mean()                         # lag = 0：对数轴画不出位置

# 分段（含左不含右），从小到大
SEG = [
    ("0",           s == 0),
    ("1 分钟内",    (s > 0) & (s < 60)),
    ("1–10 分钟",   (s >= 60) & (s < 600)),
    ("10–60 分钟",  (s >= 600) & (s < 3600)),
    ("1–24 小时",   (s >= 3600) & (s < 86400)),
    ("1 天以上",    s >= 86400),
]
seg = pd.DataFrame({"name": [k for k, _ in SEG],
                    "cnt": [int(m.sum()) for _, m in SEG]})
seg["share"] = seg["cnt"] / n
assert seg["cnt"].sum() == n, "分段没有覆盖全部记录"

# ECDF（按唯一值聚合，53 万行也很快）
vals, cnt = np.unique(s.to_numpy(), return_counts=True)
cum = np.cumsum(cnt) / n
pos = vals > 0
x_pos, y_pos = vals[pos], cum[pos]
XMIN = 10 ** np.floor(np.log10(x_pos.min()))
XMAX = s.max() * 1.6

# ---- 标题与副标题 --------------------------------------------
title = f"lag 中位 {fmt_dur(p50)}，99% 在 {fmt_dur(p99)}内"
if len(title + f"；均值 {fmt_dur(mean)}被长尾拉高") <= 40:   # 放得下才加均值
    title += f"；均值 {fmt_dur(mean)}被长尾拉高"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"

subtitle = (f"原始信令 {len(s_all):,} 行（清洗前），不含 {neg:,} 条差值小于 0 的记录，"
            f"共 {n:,} 行；lag = {LAG_DEF}，单位秒\n"
            f"左图横轴为对数刻度；lag = 0 的记录（{fmt_pct(z0)}）画不出位置，"
            f"已计入累计比例，并在右图单列")

# ---- 画布：gridspec 只写 width_ratios，边距交给 finish -------
fig = plt.figure(figsize=(10, 5.6))
gs = fig.add_gridspec(1, 2, width_ratios=[2.7, 1.3])
ax, axb = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

# ============ 左：累计分布 ============
xs = np.r_[XMIN, x_pos]
ys = np.r_[z0, y_pos]
ax.fill_between(xs, ys, step="post", color=MAIN_LIGHT, lw=0, zorder=1)
ax.step(xs, ys, where="post", color=MAIN, lw=2, zorder=3)

# 最长的 1%：灰色区块
ax.axvspan(p99, XMAX, color=GRAY, alpha=0.28, lw=0, zorder=0)
ax.text(np.sqrt(p99 * s.max()), 0.40,
        f"最长的 1%\n{fmt_dur(p99)} – {fmt_dur(s.max())}",
        ha="center", va="center", fontsize=9.5, color=MUTED, linespacing=1.5)

# P50 / P90 / P99：打点 + 引线 + 数字
for q, v in [(0.50, p50), (0.90, p90), (0.99, p99)]:
    ax.plot([v, v], [0, q], color=FAINT, lw=0.8, ls=(0, (2, 2)), zorder=2)
    ax.plot(v, q, "o", ms=6, color=MAIN, mec="white", mew=1.2, zorder=4)
    top = q > 0.95                      # P99 贴近 100%，标签放左侧居中，不顶出画面
    ax.annotate(f"P{q * 100:.0f}  {fmt_dur(v)}", (v, q),
                xytext=(-10, 0) if top else (-9, 7), textcoords="offset points",
                ha="right", va="center" if top else "bottom",
                fontsize=10, color=INK,
                weight="bold" if q == 0.50 else "normal")

# 均值：说明它被长尾拉高
ax.plot(mean, mean_rank, "o", ms=6, mfc="white", mec=MUTED, mew=1.4, zorder=4)
ax.annotate(f"均值 {fmt_dur(mean)}，已在第 {mean_rank * 100:.0f} 百分位\n"
            f"被长尾拉高，不代表典型值",
            (mean, mean_rank), xytext=(10, -10), textcoords="offset points",
            ha="left", va="top", fontsize=9.5, color=MUTED, linespacing=1.45,
            bbox=dict(boxstyle="square,pad=0.15", fc=MAIN_LIGHT, ec="none"),
            zorder=5)                   # 底色同填充色，盖住穿过的分位引线

TICKS = [(1, "1 秒"), (10, "10 秒"), (60, "1 分钟"), (600, "10 分钟"),
         (3600, "1 小时"), (86400, "1 天")]
TICKS = [(t, l) for t, l in TICKS if XMIN <= t <= XMAX]
ax.set_xscale("log")
ax.xaxis.set_major_locator(FixedLocator([t for t, _ in TICKS]))
ax.xaxis.set_major_formatter(FixedFormatter([l for _, l in TICKS]))
ax.xaxis.set_minor_locator(FixedLocator([]))
ax.set_xlim(XMIN, XMAX)
ax.set_ylim(0, 1.04)
ax.yaxis.set_major_locator(FixedLocator([0, 0.25, 0.5, 0.75, 1.0]))
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax.grid(axis="y", color=GRAY, lw=0.6, alpha=0.6)
ax.set_axisbelow(True)
ax.set_xlabel("lag（对数刻度）")
ax.set_title("累计分布：lag 不超过横轴值的记录占比",
             loc="left", fontsize=10.5, color=INK, pad=10)

# ============ 右：分段计数 ============
top_i = seg["cnt"].idxmax()
colors = [MAIN if i == top_i else GRAY for i in range(len(seg))]
yy = np.arange(len(seg))
axb.barh(yy, seg["share"], color=colors, height=0.62)
for i, r in seg.iterrows():
    hl = i == top_i
    axb.text(r["share"] + seg["share"].max() * 0.03, i,
             f"{r['cnt']:,}  {fmt_pct(r['share'])}",
             va="center", fontsize=9.5,
             color=INK if hl else MUTED, weight="bold" if hl else "normal")
axb.set_yticks(yy, seg["name"])
axb.invert_yaxis()
axb.set_xlim(0, seg["share"].max() * 1.75)
axb.xaxis.set_visible(False)
axb.spines[["bottom", "top", "right"]].set_visible(False)
axb.spines["left"].set_color(GRAY)
axb.tick_params(axis="y", length=0, pad=6, labelsize=10, labelcolor=INK)
axb.set_title("按时长分段的记录数（行，占比）",
              loc="left", fontsize=10.5, color=INK, pad=10)

viz.finish(fig, title, subtitle, SOURCE, FIG_NAME)
plt.show()

同一记录中 timestamp 与 timestamp1 的差值中位数为 23 毫秒，99% 在 1.3 秒以内（图 D2_05），对分钟级分析没有影响，统一采用 timestamp 作为事件时间（DQ-07）。

In [ ]:
pd.read_sql("SELECT * FROM dq_check_result", con)

## 信令多久来一条：观测稀疏性

同一个人相邻两条信令间隔多久？每个小时有多少人至少有一条信令

In [ ]:
s = pd.read_sql("SELECT imsi, ts FROM dwd_signal",
        con, parse_dates="ts").sort_values(['imsi','ts'])

In [ ]:
from matplotlib.ticker import FixedLocator, FixedFormatter, PercentFormatter

# 来源行：脚本名、配置版本按实际填写（第 1 节要求：数据来源 + 脚本 + 配置版本）
SOURCE = f"数据来源：沈阳移动信令样本（脱敏，2018 年国庆）；制图: {config.AUTHOR}"

def fmt_pct(p, digits=0):
    """第 6 节：一般取整；很小的值自动多保留位数（0.01%，不写 0%）。"""
    if p == 0:
        return "0%"
    while round(p * 100, digits) == 0 and digits < 4:
        digits += 1
    return f"{p:.{digits}%}"


def fmt_dur(minutes):
    """第 6 节：单位写全；超过 2 小时改用小时。"""
    if minutes < 1:
        return f"{minutes * 60:.0f} 秒"
    if minutes < 10:
        return f"{minutes:.1f} 分钟"
    if minutes <= 120:
        return f"{minutes:,.0f} 分钟"
    return f"{minutes / 60:.1f} 小时"


# %% 1 口径检查 + 分组计算
# 每小时人数按“小时”分组；数据跨多天时同一小时会把不同天的人混在一起，必须先按日期分组
n_days = s["ts"].dt.normalize().nunique()
assert n_days == 1, f"数据跨 {n_days} 天：请改为按 日期+小时 分组"
DAY_LABEL = s["ts"].dt.strftime("%m-%d").iloc[0]

# diff 依赖行顺序：先按 人、时间 排序，再在每个人内部做差，保证间隔不跨人、不出负数
s = s.sort_values(["imsi", "ts"])
gap = (s.groupby("imsi")["ts"].diff().dt.total_seconds() / 60).dropna()
assert (gap >= 0).all()

n_user = s["imsi"].nunique()
n_rec = len(s)
n_gap = len(gap)
p50, p90, p99 = gap.quantile([0.5, 0.9, 0.99])
share_zero = (gap == 0).mean()          # 同一时刻的重复记录
share_gt60 = (gap > 60).mean()          # 按“条数”算：超过 1 小时的间隔占比
time_gt60 = gap[gap > 60].sum() / gap.sum()  # 按“时间”算：这些长间隔覆盖的时长占比

hourly = (s.assign(h=s["ts"].dt.hour)
           .groupby("h")["imsi"].nunique()
           .reindex(range(24), fill_value=0)
           .rename("人数").to_frame())
hourly["占比"] = hourly["人数"] / n_user   # 不先四舍五入，展示时再格式化


# %% 2 图一：相邻信令间隔的累计分布（ECDF，对数横轴）
x_sorted = np.sort(gap.to_numpy())
x_min = 1 / 60                                   # 1 秒
grid = np.geomspace(x_min, 24 * 60, 600)
ecdf = np.searchsorted(x_sorted, grid, side="right") / n_gap   # 0 秒间隔计入曲线起点

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(grid, ecdf, color=MAIN, lw=2.2)
ax.set_xscale("log")
ax.set_xlim(x_min, 24 * 60)
ax.set_ylim(0, 1.02)

# 横轴标实际时长，不标 10^n
ticks = [1/60, 10/60, 1, 5, 15, 60, 6 * 60, 24 * 60]
labels = ["1 秒", "10 秒", "1 分钟", "5 分钟", "15 分钟", "1 小时", "6 小时", "24 小时"]
ax.xaxis.set_major_locator(FixedLocator(ticks))
ax.xaxis.set_major_formatter(FixedFormatter(labels))
ax.xaxis.set_minor_locator(FixedLocator([]))
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.set_xlabel("同一人相邻两条信令的间隔（对数刻度）")
ax.set_ylabel("间隔不超过该值的占比")
ax.grid(axis="y", color=GRAY, lw=0.6, alpha=0.6)

# 1 小时参考线：和“每小时一个位置”的口径直接相关
ax.axvline(60, color=GRAY_DARK, lw=1, ls="--")
ax.text(60 * 1.15, 0.06, f"超过 1 小时的间隔\n按条数占 {fmt_pct(share_gt60, 1)}\n"
        f"按时长占 {fmt_pct(time_gt60)}", color=MUTED, fontsize=9.5, va="bottom")

# 三个分位点：打点 + 直接标注（不用图例）
for q, v, txt, dx, dy, ha, va in [
    (0.50, p50, f"一半的间隔 ≤ {fmt_dur(p50)}", 10, -4, "left", "top"),
    (0.90, p90, f"90% ≤ {fmt_dur(p90)}", -10, 4, "right", "bottom"),
    (0.99, p99, f"99% ≤ {fmt_dur(p99)}", 10, -6, "left", "top"),
]:
    ax.plot(v, q, "o", color=MAIN_DARK, ms=6, zorder=3)
    ax.annotate(txt, (v, q), xytext=(dx, dy), textcoords="offset points",
                ha=ha, va=va, fontsize=10, color=INK)

title1 = f"同一人相邻信令中位间隔 {fmt_dur(p50)}，{fmt_pct(share_gt60, 1)} 的间隔超过 1 小时"
subtitle1 = (f"{DAY_LABEL} 共 {n_user:,} 人、{n_gap:,} 个间隔；同一人按时间排序后相邻两条相减；"
             f"\n0 秒间隔（同一时刻重复记录）占 {fmt_pct(share_zero, 1)}，计入曲线起点；"
             f"信令随手机活动产生，间隔长不等于人没动")
assert len(title1) <= 40, len(title1)
viz.finish(fig, title1, subtitle1, SOURCE, "Q01_信令间隔分布.png")

# %% 3 图二：每小时至少有 1 条信令的人数占比（折线）
peak_h = hourly["人数"].idxmax()
low_h = hourly["人数"].idxmin()
pk, lo = hourly.loc[peak_h], hourly.loc[low_h]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(hourly.index, hourly["占比"], color=MAIN, lw=2.2, marker="o", ms=3.5)
ax.fill_between(hourly.index, hourly["占比"], 1, color=MAIN_LIGHT, alpha=0.6, lw=0)
ax.set_xlim(-0.5, 23.5)
ax.set_ylim(0, 1.0)
ax.set_xticks(range(0, 24, 3))
ax.set_xticklabels([f"{h} 点" for h in range(0, 24, 3)])
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.set_xlabel(f"小时（{DAY_LABEL}）")
ax.set_ylabel("该小时至少有 1 条信令的人数占比")
ax.grid(axis="y", color=GRAY, lw=0.6, alpha=0.6)

# 峰值、谷值：打点 + 写人数和占比
for h, row, dy, va in [(peak_h, pk, -12, "top"), (low_h, lo, -12, "top")]:
    ax.plot(h, row["占比"], "o", color=MAIN_DARK, ms=7, zorder=3)
    ax.annotate(f"{h} 点：{row['人数']:,.0f} 人（{fmt_pct(row['占比'])}）", (h, row["占比"]),
                xytext=(0, dy), textcoords="offset points",
                ha="center", va=va, fontsize=10, color=INK)

# 浅色区域 = 当小时看不到的人
ax.text(11.5, 0.97, "浅色区域：该小时没有任何信令、看不到位置的人",
        ha="center", va="top", fontsize=9.5, color=MUTED)

night = "凌晨 " if low_h < 6 else ""
title2 = (f"{peak_h} 点能看到的人最多，占 {fmt_pct(pk['占比'])}；"
          f"{night}{low_h} 点最少，仅 {fmt_pct(lo['占比'])}")
subtitle2 = (f"{DAY_LABEL} 分母为当天出现过的 {n_user:,} 人；某小时内至少有 1 条信令即算“能看到”"
             f"\n没有信令≠不在场：夜间偏低多因手机静默，不能读作人少；此图不是在场人数")
assert len(title2) <= 40, len(title2)
viz.finish(fig, title2, subtitle2, SOURCE, "Q02_每小时可观测人数.png")

plt.show()